In [3]:
from bitarray import bitarray
import hashlib
from hashlib import sha3_256, sha256, blake2b
import math 
import mmh3
import string
import json
import requests

In [ ]:
# Implement using bitarray library and fixed number of bits to store word data
#asked ChatGPT how large my bitarray needs to be to hold the large dataset
#isn't this very large and will cause hash collisions?

size = 360000
bits = bitarray(size)
bits.setall(0)
# bits n should be larger than data set size

In [8]:
#Implement filter
# Asked ChatGPT what to edit to change bit size and hashes

class BloomFilter:
    def __init__(self, size, hash_functions): #now uses any hash function
        #items_count number of items expected to be sotred in bloom filter
        self.size = size #set size asspecified in bloom
        self.bit_array = bitarray(self.size) #bit array of given size
        self.bit_array.setall(0) #initialize all bits as 0
        self.hash_functions = hash_functions #allows provided hashing
    def add(self, item):
        for func in self.hash_functions:
            digest = func(item) % self.size #each hash produces integer from item within bounds of bit array
            self.bit_array[digest] = 1 #then sets that bit to 1
    def check(self, item):
        return all(self.bit_array[func(item) % self.size] for func in self.hash_functions) #checks if item is in filter
    
# hashes do not have to change

In [1]:
# call in words. Strip blank space and append individually to words list
with open('words.txt', 'r') as file:
    for line in file:
        word = line.strip().lower()
    word_set = set(words)

FileNotFoundError: [Errno 2] No such file or directory: 'words.txt'

In [18]:
# Implement a function to test how well the Bloom filter suggests corrections. 
# A suggestion list is considered "good" if it contains no more than three suggestions and includes the correct word.

#https://www.geeksforgeeks.org/python/read-json-file-using-python/

#call in typos and visualize
with open('typos.json', 'r') as file2:
    typos = json.load(file2)

In [10]:
# For each word in the word list, apply all three hash functions
# set corresponding bits in bitarray to 1

def my_hash(s):
    return int(sha256(s.lower().encode()).hexdigest(), 16) % size
def my_hash2(s):
    return int(blake2b(s.lower().encode()).hexdigest(), 16) % size
def my_hash3(s):
    return int(sha3_256(s.lower().encode()).hexdigest(), 16) % size


for word in words: 
    bits[my_hash(word)] = 1
    bits[my_hash2(word)] = 1
    bits[my_hash3(word)] = 1 #set bits to 1 in each hash when word is presenT
    

In [11]:
#b. Create a function that checks all possible single-character substitutions for a given word using the Bloom filter. 
# Return words flagged by the filter as potential matches. 

#I need the function to take in each word, replace a single character, compare to the rest of list and return if it is a match. Repeat for all letter combinations

#Asked ChatGPT how to make a function that replces single character in given word
def single_char_changes(word): 
    replaced_words = []
    for i in range(len(word)):
        for letter in string.ascii_lowercase:
            if word[i] != letter:
                changed = word[:i] + letter + word[i+1:]
                replaced_words.append(changed)
    return replaced_words

#Tested that the function does single character substitution for word given
print(single_char_changes('cat'))
len(single_char_changes('cat'))

['aat', 'bat', 'dat', 'eat', 'fat', 'gat', 'hat', 'iat', 'jat', 'kat', 'lat', 'mat', 'nat', 'oat', 'pat', 'qat', 'rat', 'sat', 'tat', 'uat', 'vat', 'wat', 'xat', 'yat', 'zat', 'cbt', 'cct', 'cdt', 'cet', 'cft', 'cgt', 'cht', 'cit', 'cjt', 'ckt', 'clt', 'cmt', 'cnt', 'cot', 'cpt', 'cqt', 'crt', 'cst', 'ctt', 'cut', 'cvt', 'cwt', 'cxt', 'cyt', 'czt', 'caa', 'cab', 'cac', 'cad', 'cae', 'caf', 'cag', 'cah', 'cai', 'caj', 'cak', 'cal', 'cam', 'can', 'cao', 'cap', 'caq', 'car', 'cas', 'cau', 'cav', 'caw', 'cax', 'cay', 'caz']


75

In [ ]:
#BloomFilter(number of items, false positive)
#https://www.geeksforgeeks.org/python/bloom-filters-introduction-and-python-implementation/

#add words to BloomFilter using all three hash functions
three_hash_functions = [my_hash, my_hash2, my_hash3]
two_hash_functions = [my_hash, my_hash2]
one_hash_function = [my_hash]

bloomf_3 = BloomFilter(1000000,three_hash_functions)
bloomf_2 = BloomFilter(1000000, two_hash_functions)
bloomf_1 = BloomFilter(1000000, one_hash_function)

for word in words:
    bloomf_3.add(word)
    bloomf_2.add(word)
    bloomf_1.add(word)



In [ ]:
# make list of all words with replace all characters
# for each word in that list, check if its in the bloom
# return amount of words from list that are in bloom (possible suggestions)
# return if correct word is in that list and how many words returned as possible suggestions
# if possible suggestions is more than 3, bad outcome. If less than 3, good outcome

def good_bloom(typed_word, bloom):
  good = 0
  total = len(typo_pairs)
  misidentified = 0 
  def single_char_changes(typed_word): 
        replaced_words_bloom = []
        for i in range(len(typed_word)):
            for letter in string.ascii_lowercase:
                if typed_word[i] != letter:
                    changed = typed_word[:i] + letter + typed_word[i+1:]
                    replaced_words_bloom.append(changed)
        return replaced_words_bloom
  candidates = single_char_changes(typed_word) # puts all single character changes in list as candidates to be a match
  matches = []
  for candidate in candidates: #cycles through each single character change and stores in matches if it is in the word list
    #iterate all candidates into hashes then check the bloom for those hashes
    if bloom.check(candidate):
        matches.append(candidate)
    #Do I need to return all candidates' hashes back into strings here so it can determine if its a good/bad outcome
  print(matches)
  if len(matches) <= 3 and correct_word in matches: #and correct_word in matches: # is matches is more than 3, deems it a bad outcome. If matches is less than 3 deems it a good outcome; need to add if it includes correct word is good, if not is bad. 
      return (f"This is a good outcome, it has {len(matches)} matches and includes the correct word)")
  else:
      return ("This is a bad outcome")
  

In [31]:
good_bloom('floeer', 'flower', bloomf_1)

['aloeer', 'bloeer', 'cloeer', 'dloeer', 'eloeer', 'gloeer', 'hloeer', 'iloeer', 'jloeer', 'kloeer', 'lloeer', 'mloeer', 'oloeer', 'ploeer', 'qloeer', 'rloeer', 'sloeer', 'tloeer', 'uloeer', 'vloeer', 'wloeer', 'xloeer', 'yloeer', 'zloeer', 'faoeer', 'fboeer', 'fcoeer', 'fdoeer', 'feoeer', 'ffoeer', 'fgoeer', 'fhoeer', 'fioeer', 'fkoeer', 'fmoeer', 'fnoeer', 'fooeer', 'fpoeer', 'fqoeer', 'froeer', 'fsoeer', 'ftoeer', 'fuoeer', 'fvoeer', 'fwoeer', 'fxoeer', 'fyoeer', 'fzoeer', 'flaeer', 'flbeer', 'flceer', 'fldeer', 'fleeer', 'flfeer', 'flgeer', 'flheer', 'flieer', 'fljeer', 'flkeer', 'flleer', 'flmeer', 'flneer', 'flqeer', 'flreer', 'flseer', 'flteer', 'flueer', 'flveer', 'flweer', 'flxeer', 'flyeer', 'flzeer', 'floaer', 'flober', 'flocer', 'floder', 'flofer', 'floger', 'floher', 'floier', 'flojer', 'floker', 'floler', 'flomer', 'floner', 'flooer', 'floper', 'floqer', 'florer', 'floser', 'floter', 'flouer', 'flover', 'flower', 'floxer', 'floyer', 'flozer', 'floear', 'floebr', 'floecr',

'This is a bad outcome'